# Lab 3.2. The acquisition pipeline: one year of Retiro, by script

**Module 2, Session 3. ACS-UPM Diploma in Engineering, Data Science and Artificial Intelligence**

By the end of this notebook you should be able to:

1. Build a request to an unfamiliar API from its documentation, with the key kept out of the code.
2. Run the whole chain: call, raw copy with a timestamp, schema check, typed table, Parquet, provenance note.
3. Follow a paginated endpoint to the end, and back off correctly when a service answers 429.

Twenty-four minutes. The service is AEMET OpenData, the Spanish meteorological agency; the
resource is the daily climatology of the Retiro station (indicativo 3195) for 2025.

## 0. Setup

This cell locates the course folder wherever the notebook runs: locally, in Colab from the
course repository, or in Colab from the shared Drive folder. Run it first.

In [ ]:
from pathlib import Path

REPO = "https://github.com/antiafer/acs-upm-mod2-s01.git"   # course repository
DRIVE = "ACS-UPM/Mod2-S01"                                 # folder inside My Drive

def course_folder():
    """Return the folder that contains data/, wherever we are running."""
    here = Path.cwd()
    for base in (here, here.parent):
        if (base / "data").is_dir():
            return base
    try:
        import google.colab                            # noqa: F401
    except ImportError:
        raise FileNotFoundError("No data/ folder next to the notebook or one level up.")
    root = Path("/content/acs-mod2")
    if not (root / "data").is_dir():
        import subprocess
        subprocess.run(["git", "clone", "-q", REPO, str(root)], check=False)
    if (root / "data").is_dir():
        return root
    from google.colab import drive
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / DRIVE
    if (root / "data").is_dir():
        return root
    raise FileNotFoundError("Could not find the course folder.")

BASE = course_folder()
DATA = BASE / "data"
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
RAW = WORK / "raw"; RAW.mkdir(exist_ok=True)
print("Course folder:", BASE)
print("Raw copies go to:", RAW)

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, module=None):
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for pkg, mod in [("requests", None), ("jsonschema", None), ("beautifulsoup4", "bs4"),
                 ("lxml", None), ("pyarrow", None)]:
    ensure(pkg, mod)

import json, re, hashlib, time, datetime as dt
import pandas as pd, requests
print("pandas", pd.__version__, "| requests", requests.__version__)

### Live or cached

Every call in this notebook first tries the live service. If the service does not answer
(twenty people share one address in the room, and public services rate-limit), the cell loads a
copy of the same response saved in `data/`, and says so. The raw copy is not a fallback only: it
is the first step of the pipeline, and the reason a result can be reproduced next year.

**The cached copies are synthetic.** They reproduce the exact shape of each service's answer
(envelope, field names, numbers as strings, missing keys) but the values are invented for the
exercise. Read any number you compute from them as an exercise result, not as a fact about Madrid.
`SOURCES_session3.md` lists what is faithful and what is not in each file.


In [ ]:
def fetch_or_cache(url, cache, *, headers=None, params=None, timeout=15):
    """GET a URL; on any failure, load the cached response from data/. Returns (json, source)."""
    try:
        r = requests.get(url, headers=headers, params=params, timeout=timeout)
        r.raise_for_status()
        doc = r.json()
        # HTTP 200 does not mean the call worked. AEMET answers 200 and puts the real
        # status inside the body, so an expired download link arrives as a normal
        # response reading {"descripcion": "datos expirados", "estado": 404}. Without
        # this check the error envelope flows on as if it were the data.
        if isinstance(doc, dict) and doc.get("estado") not in (None, 200):
            raise ValueError(f"envelope reports estado {doc['estado']}: {doc.get('descripcion')}")
        stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        (RAW / f"{cache.stem}_{stamp}.json").write_text(json.dumps(doc, ensure_ascii=False), encoding="utf-8")
        return doc, "live"
    except Exception as e:                            # noqa: BLE001
        print(f"Live call failed ({type(e).__name__}); using the cached copy {cache.name}")
        return json.loads(cache.read_text(encoding="utf-8")), "cache"

print("fetch_or_cache ready")

## 1. The key stays out of the code

The key is read from an environment variable or from a local file listed in `.gitignore`.
If neither exists, the notebook runs from the cached responses and says so. Never paste a key
into a cell: a notebook is shared, a key is a password.

In [ ]:
import os
KEY = os.environ.get("AEMET_KEY") or (
    (BASE / "aemet_key.txt").read_text().strip() if (BASE / "aemet_key.txt").exists() else None)
MODE = "live" if KEY else "cached"
print("Mode:", MODE, "" if KEY else "(no key found: set AEMET_KEY or create aemet_key.txt next to data/)")

### Exercise 1. Build the request from the documentation

The AEMET documentation gives the resource as

```
/api/valores/climatologicos/diarios/datos/fechaini/{fechaIniStr}/fechafin/{fechaFinStr}/estacion/{idema}
```

with dates in the form `2025-01-01T00:00:00UTC` and the key sent in a header named `api_key`.
Build `url` for the whole of 2025 at station 3195, and `headers`. Do not call anything yet.

In [ ]:
HOST = "https://opendata.aemet.es/opendata"
url = (f"{HOST}/api/valores/climatologicos/diarios/datos"
       f"/fechaini/2025-01-01T00:00:00UTC/fechafin/2025-12-31T23:59:59UTC/estacion/3195")
headers = {"api_key": KEY or "CACHED", "accept": "application/json"}
print(url)

In [ ]:
assert url.startswith(HOST + "/api/valores/climatologicos/diarios/datos/fechaini/2025-01-01T00:00:00UTC")
assert url.endswith("/estacion/3195") and "fechafin/2025-12-31" in url
assert "api_key" in headers
print("Checks passed: the request is built from the documentation, before any call.")

## 2. The call, in two steps

### Exercise 2. Step 1: the envelope

AEMET does not return the data at once. The first call returns a small JSON whose `datos` field
holds a second URL, and the data are fetched from there. This is documented behaviour; finding it
in the documentation rather than being surprised by it is the skill.

Make the first call with `fetch_or_cache` (cache: `aemet_step1_response.json`), and extract the
`datos` URL into `data_url`.

In [ ]:
step1, source1 = fetch_or_cache(url, DATA / "aemet_step1_response.json", headers=headers)
data_url = step1["datos"]
print("Source:", source1)
print(json.dumps(step1, indent=1))

In [ ]:
assert step1.get("estado") == 200 and data_url.startswith("https://opendata.aemet.es/")
print("Checks passed: the envelope says exito and points to the data.")

### Exercise 3. Step 2: the data, and the raw copy first

Fetch `data_url` (cache: `aemet_retiro_2025_daily.json`) into `records`. `fetch_or_cache` has
already written a timestamped copy under `raw/` if the call was live. Confirm what arrived:
how many records, and what one looks like.

In [ ]:
records, source2 = fetch_or_cache(data_url, DATA / "aemet_retiro_2025_daily.json", headers=headers)
print("Source:", source2, "| records:", len(records))
print(json.dumps(records[0], indent=1, ensure_ascii=False))

In [ ]:
assert isinstance(records, list) and len(records) == 365
assert isinstance(records[0]["tmed"], str), "Look at the types: AEMET sends numbers as strings"
print("Checks passed. Notice: every number is a string with a decimal comma. Nothing is typed yet.")

## 3. Nothing enters without passing the schema

### Exercise 4. Validate before touching

`aemet_daily_schema.json` describes one daily record: which fields are required and what shape
the numeric strings must have. Validate every record and count the failures.

**Predict before you run:** three days in the series have no temperature at all (the station did
not measure). Will the schema reject them? Read the `required` list before answering.

In [ ]:
from jsonschema import Draft202012Validator
schema = json.loads((DATA / "aemet_daily_schema.json").read_text(encoding="utf-8"))
validator = Draft202012Validator(schema)
failures = [(r["fecha"], [e.message for e in validator.iter_errors(r)]) for r in records]
failures = [f for f in failures if f[1]]
print("Records failing the schema:", len(failures))
print("Days without tmed:", sum(1 for r in records if "tmed" not in r))

In [ ]:
assert len(failures) == 0
assert sum(1 for r in records if "tmed" not in r) == 3
print("Checks passed. The schema requires only what is truly required; a missing measurement is")
print("legitimate and stays visible as a gap. A wrong shape in a present field would be rejected.")

## 4. The typed table

### Exercise 5. From strings with commas to numbers

Build `daily` from `records`: parse `fecha` as a date, convert `tmed`, `tmin`, `tmax`, `prec`
to floats (decimal comma), and treat `prec == "Ip"` (inapreciable, below 0.1 mm) as 0.0 while
keeping a boolean column `prec_trace` that remembers it.

In [ ]:
daily = pd.DataFrame(records)
daily["fecha"] = pd.to_datetime(daily["fecha"])
daily["prec_trace"] = daily["prec"].eq("Ip")
for c in ["tmed", "tmin", "tmax", "prec"]:
    daily[c] = pd.to_numeric(daily[c].replace("Ip", "0,0").str.replace(",", "."), errors="coerce")
daily = daily.sort_values("fecha").reset_index(drop=True)
print(daily.dtypes[["fecha", "tmed", "prec", "prec_trace"]])
daily[["fecha", "tmed", "tmin", "tmax", "prec", "prec_trace"]].head(3)

In [ ]:
assert daily["tmed"].dtype.kind == "f" and daily["fecha"].dtype.kind == "M"
assert daily["tmed"].isna().sum() == 3, "The three unmeasured days must stay NaN, not 0"
assert abs(daily["prec"].sum() - 529.6) < 0.2, f"Annual precipitation off: {daily['prec'].sum():.1f}"
wettest = daily.loc[daily["prec"].idxmax()]
print(f"Checks passed. 2025 at Retiro: {daily['prec'].sum():.1f} mm; wettest day "
      f"{wettest['fecha'].date()} with {wettest['prec']:.1f} mm.")

### Exercise 6. Parquet and the provenance note

Save `daily` as Parquet in `WORK` and write the five-line provenance note as a dictionary:
what, where (the resource URL), when (now, UTC), licence, one caveat. The caveat is the point:
write the one thing a reader must know before trusting the file.

In [ ]:
daily.to_parquet(WORK / "retiro_2025_daily.parquet", index=False)
provenance = {
    "what": "Daily climatology, station 3195 Madrid Retiro, 2025-01-01 to 2025-12-31",
    "where": url,
    "when": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
    "licence": "AEMET OpenData, reuse permitted with attribution (Ley 37/2007; AEMET legal notice)",
    "caveat": f"Numbers arrive as strings with a decimal comma; 'Ip' means trace and was set to 0.0 "
              f"({int(daily['prec_trace'].sum())} days); three days have no temperature. Source: {source2}.",
}
(WORK / "retiro_2025_provenance.json").write_text(json.dumps(provenance, indent=1, ensure_ascii=False))
for k, v in provenance.items():
    print(f"{k:<8} {v}")

In [ ]:
assert (WORK / "retiro_2025_daily.parquet").exists()
assert all(str(v).strip() for v in provenance.values()) and set(provenance) == {"what", "where", "when", "licence", "caveat"}
print("Checks passed: raw copy, schema, typed table, Parquet and note. The chain is complete.")

## 5. Two things every real API will do to you

### Exercise 7. Pagination

Most portals return results in pages: a page of items and a `next` link, `null` on the last page.
`stations_page_1.json` to `_3.json` are the three pages of a datos.madrid.es-style endpoint.

Write `fetch_all(first_url)` that follows `next` until it is `null`, collecting every item of
`@graph`. The helper `page_cache` maps each page URL to its cached file, so the loop runs offline.

In [ ]:
def page_cache(page_url):
    """The cached file for a page URL (?page=N); page 1 has no parameter."""
    m = re.search(r"page=(\d+)", page_url)
    return DATA / f"stations_page_{m.group(1) if m else 1}.json"

import re
first_url = "https://datos.madrid.es/egob/catalogo/aforos.json"

def fetch_all(start):
    items, page_url, pages = [], start, 0
    while page_url:
        page, _ = fetch_or_cache(page_url, page_cache(page_url))
        items.extend(page["@graph"])
        page_url = page.get("next")
        pages += 1
    return items, pages

items, pages = fetch_all(first_url)
print(f"{len(items)} items over {pages} pages")

In [ ]:
assert len(items) == 63 and pages == 3, "Follow next until it is null: 63 items, 3 pages"
assert len({it["id"] for it in items}) == 63, "No item should be collected twice"
print("Checks passed. Stopping at page 1 gives 25 items and no error: the pagination version of the split table.")

### Exercise 8. Backing off from a 429

When a service answers **429 Too Many Requests**, the correct reaction is to wait and retry,
waiting longer each time, and to respect the `Retry-After` header if the service sends one.
`FlakyService` below simulates a service that answers 429 twice and then 200. Write
`get_with_backoff(service, url)` that retries up to `max_tries` times with exponential waits.

In [ ]:
class FlakyService:
    """Answers 429 (with Retry-After: 1) twice, then 200. Sleeps are shortened for the exercise."""
    def __init__(self, failures=2):
        self.failures, self.calls = failures, 0
    def get(self, url):
        self.calls += 1
        if self.calls <= self.failures:
            return type("R", (), {"status_code": 429, "headers": {"Retry-After": "1"}, "json": lambda s: None})()
        return type("R", (), {"status_code": 200, "headers": {}, "json": lambda s: {"ok": True, "url": url}})()

def get_with_backoff(service, url, max_tries=5, base=0.05):
    waits = []
    for attempt in range(max_tries):
        r = service.get(url)
        if r.status_code == 200:
            return r.json(), waits
        if r.status_code == 429:
            wait = float(r.headers.get("Retry-After", 0)) * base or base * 2 ** attempt
            waits.append(wait); time.sleep(wait)
            continue
        raise RuntimeError(f"Unexpected status {r.status_code}")
    raise RuntimeError("Gave up after retries")

svc = FlakyService()
result, waits = get_with_backoff(svc, "https://example.org/resource")
print("Calls:", svc.calls, "| waits:", waits, "| result:", result)

In [ ]:
assert result == {"ok": True, "url": "https://example.org/resource"}
assert svc.calls == 3 and len(waits) == 2, "Two 429s, two waits, success on the third call"
print("Checks passed. Hammering a service after a 429 gets you banned; waiting gets you the data.")

### Extension A. A live call without a key: Overpass

Overpass answers queries over OpenStreetMap without registration. The query below asks for
bridges in a bounding box of central Madrid. Run it live if the room's connection allows; the
cached response is the one Lab 3.1 flattened. Overpass is a donated public service: one call per
person, not one per cell.

It also refuses anonymous clients: a request with the default `python-requests` User-Agent gets
*406 Not Acceptable*. Say who you are in a `User-Agent` header, as any polite client should.

In [ ]:
query = """[out:json][timeout:25];
way["bridge"="yes"](40.395,-3.745,40.435,-3.665);
out tags;"""
try:
    UA = "ACS-UPM-course/1.0 (teaching)"
    r = requests.post("https://overpass-api.de/api/interpreter", data={"data": query},
                      headers={"User-Agent": UA}, timeout=30)
    r.raise_for_status(); ov, src = r.json(), "live"
except Exception as e:                                                        # noqa: BLE001
    print(f"Live call failed ({type(e).__name__}); using the cached copy")
    ov, src = json.loads((DATA / "overpass_bridges_madrid.json").read_text(encoding="utf-8")), "cache"
print(src, "|", len(ov["elements"]), "ways tagged bridge=yes")

---

## Take-aways

- The request is built from the documentation before any call; the key lives outside the code.
- Raw copy first, with a timestamp. A result that cannot be re-obtained cannot be defended.
- A schema at the door: valid JSON is not yet usable data.
- Numbers arrive as strings, with commas, with sentinels such as `Ip`. Typing is a step, not an afterthought.
- Pagination and 429 are not edge cases; they are what every real endpoint does.

**References.** AEMET OpenData documentation. IETF RFC 9110 (HTTP Semantics), sections on status
codes and Retry-After. Reitz, *Requests: HTTP for Humans*, documentation.